# Caso Técnico: Analista de Datos
## Punto 1 — Procesamiento del extracto crudo (`prueba.txt`)

Este notebook cubre los tres entregables del punto 1 del caso:

1. **Limpieza estructural** del archivo crudo (delimitador, espacios, columnas nulas en los extremos).
2. **Parsing de JSON** embebido para extraer el motivo de los mensajes `failed`.
3. **Clasificación ligera (NLP por palabras clave)** de los mensajes `incoming`.

> **Nota sobre el archivo de entrada:** `prueba.txt` no es un CSV estándar. Está delimitado por `|`, cada
> celda viene rellena con espacios (padding de ancho fijo), y las columnas `content_attributes` /
> `additional_attributes` traen JSON con comillas internas escapadas (`\"`). Por eso `pd.read_csv` normal
> falla — se necesita el módulo `csv` con un manejo explícito de comillas y escape.

In [75]:
import csv
import json
import pandas as pd

RUTA_ARCHIVO = "Data/Input/prueba.txt"  # ajusta la ruta si tu archivo está en otra carpeta (ej. "data/prueba.txt")

pd.set_option("display.max_colwidth", 80)

## 1. Limpieza Estructural

Pasos que resuelve esta sección:

- Carga el archivo respetando el delimitador `|` y las comillas escapadas del JSON interno (usando
  `csv.reader` en vez de `pandas.read_csv`, que truena con este formato).
- Descarta la fila separadora tipo Markdown (`|------|------|...|`) que trae el extracto.
- Elimina la columna vacía que se genera a la izquierda y a la derecha por el `|` inicial/final de cada línea.
- Quita los espacios de relleno (padding) de cada celda.
- Convierte a tipos correctos: fechas (`created_at`, `updated_at`), booleanos (`private`) y los IDs
  numéricos (que vienen con `.` como separador de miles, ej. `31.888` → `31888`).

In [76]:
def cargar_dataframe(ruta: str) -> pd.DataFrame:
    """Carga el extracto crudo respetando el delimitador '|' y las comillas
    escapadas del JSON embebido en content_attributes / additional_attributes."""
    with open(ruta, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f, delimiter="|", quotechar='"',
                             escapechar="\\", doublequote=False)
        filas = list(reader)

    # La primera fila es el encabezado; quitamos la columna vacía de los extremos
    encabezado = [c.strip() for c in filas[0]][1:-1]

    filas_datos = []
    for r in filas[1:]:
        primer_celda = r[1].strip()
        # Descarta la fila separadora estilo Markdown: |------|------|...
        if primer_celda != "" and set(primer_celda) <= {"-"}:
            continue
        filas_datos.append([c.strip() for c in r[1:-1]])

    df = pd.DataFrame(filas_datos, columns=encabezado)
    df = df.replace("", pd.NA)  # celdas vacías -> NA real, no string vacío
    return df


def convertir_tipos(df: pd.DataFrame) -> pd.DataFrame:
    """Castea columnas a su tipo real: IDs numéricos, fechas y booleanos."""
    columnas_id = [
        "id", "account_id", "inbox_id", "conversation_id", "sender_id",
        "contact_id", "sistema_lead_id", "conversation_display_id",
        "id_sistema", "id_db_externa", "id_leads_db_externa",
    ]
    for col in columnas_id:
        # Los IDs vienen con '.' como separador de miles (ej. "31.888" == 31888)
        limpio = df[col].astype("string").str.replace(".", "", regex=False)
        df[col] = pd.to_numeric(limpio, errors="coerce").astype("Int64")

    for col in ["created_at", "updated_at"]:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    df["private"] = df["private"].map({"true": True, "false": False})
    return df


df = cargar_dataframe(RUTA_ARCHIVO)
df = convertir_tipos(df)

print(f"Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}")
df.dtypes

Filas: 3,687  |  Columnas: 24


id                                    Int64
content                              object
account_id                            Int64
inbox_id                              Int64
conversation_id                       Int64
message_type                         object
created_at                   datetime64[ns]
updated_at                   datetime64[ns]
private                                bool
status                               object
source_id                            object
content_type                         object
content_attributes                   object
sender_type                          object
sender_id                             Int64
additional_attributes                object
processed_message_content            object
sentiment                            object
contact_id                            Int64
sistema_lead_id                       Int64
conversation_display_id               Int64
id_sistema                            Int64
id_db_externa                   

In [77]:
# Vista rápida de nulos por columna, para detectar campos poco confiables
df.isna().sum().sort_values(ascending=False)

id_leads_db_externa          3687
id_db_externa                3687
source_id                    2435
sender_type                   750
sender_id                     750
processed_message_content     119
content                       119
id_sistema                      0
conversation_display_id         0
sistema_lead_id                 0
contact_id                      0
sentiment                       0
additional_attributes           0
id                              0
content_type                    0
status                          0
private                         0
updated_at                      0
created_at                      0
message_type                    0
conversation_id                 0
inbox_id                        0
account_id                      0
content_attributes              0
dtype: int64

In [78]:
df.head(3)

,id,content,account_id,inbox_id,conversation_id,message_type,created_at,updated_at,private,status,...,sender_id,additional_attributes,processed_message_content,sentiment,contact_id,sistema_lead_id,conversation_display_id,id_sistema,id_db_externa,id_leads_db_externa
0,31888,La conversación fue marcada por Rojas bpo,37,78,2460,activity,2023-07-28 14:27:58,2023-07-28 14:27:58,False,sent,...,<NA>,{},La conversación fue marcada por Rojas bpo,{},15090,16734,1486,31888,<NA>,<NA>
1,30276,Gracias por comunicarse con nosotros. 💙¶¶Espero haber sido de ayuda.¶¶Le des...,37,78,5355,outgoing,2023-07-26 00:19:49,2023-07-26 00:19:51,False,delivered,...,18,{},Gracias por comunicarse con nosotros. 💙¶¶Espero haber sido de ayuda.¶¶Le des...,{},14459,16103,4265,30276,<NA>,<NA>
2,35098,"¡Hola, Johana Andrea Pascuas Perez! 💖¶🚀 ¡Llegó ! Un nuevo serie comienza con...",37,78,7226,outgoing,2023-07-29 13:43:11,2023-07-29 13:43:12,False,failed,...,16,"{""template_params"": {""name"": ""lanzamiento_"", ""category"": ""MARKETING"", ""langu...","¡Hola, Johana Andrea Pascuas Perez! 💖¶🚀 ¡Llegó ! Un nuevo serie comienza con...",{},20743,22387,6006,35098,<NA>,<NA>


## 2. Parsing de JSON — motivo de los mensajes `failed`

Al revisar el archivo real se confirma dónde vive cada dato (esto no es obvio desde el enunciado, así
que vale la pena documentarlo):

- El **error externo** (`external_error`) siempre viene en la columna **`content_attributes`**,
  como JSON tipo `{"external_error": "..."}`.
- El **nombre de la plantilla** (`template_params.name`) viene en la columna **`additional_attributes`**,
  no en `content_attributes`.

Se usa un parser de JSON "a prueba de fallos": si la celda no es JSON válido (o está vacía, `"{}"`),
regresa un diccionario vacío en vez de tronar el pipeline completo.

In [79]:
def extraer_json_seguro(texto) -> dict:
    """Parsea JSON de forma segura. Si falla o está vacío, regresa {} en vez de lanzar excepción."""
    if pd.isna(texto):
        return {}
    try:
        return json.loads(texto)
    except (json.JSONDecodeError, TypeError):
        return {}


def enriquecer_mensajes_fallidos(df: pd.DataFrame) -> pd.DataFrame:
    """Agrega columnas estructuradas 'error_externo' y 'plantilla_nombre'
    para los mensajes con status == 'failed'."""
    df["error_externo"] = pd.NA
    df["plantilla_nombre"] = pd.NA

    mask_failed = df["status"] == "failed"

    atributos_contenido = df.loc[mask_failed, "content_attributes"].apply(extraer_json_seguro)
    df.loc[mask_failed, "error_externo"] = atributos_contenido.apply(
        lambda d: d.get("external_error")
    )

    atributos_adicionales = df.loc[mask_failed, "additional_attributes"].apply(extraer_json_seguro)
    df.loc[mask_failed, "plantilla_nombre"] = atributos_adicionales.apply(
        lambda d: d.get("template_params", {}).get("name")
    )
    return df


df = enriquecer_mensajes_fallidos(df)

total_failed = (df["status"] == "failed").sum()
print(f"Mensajes failed: {total_failed}")
print(f"  con error extraído:    {df['error_externo'].notna().sum()}")
print(f"  con plantilla extraída: {df['plantilla_nombre'].notna().sum()}")

Mensajes failed: 1396
  con error extraído:    1396
  con plantilla extraída: 1395


In [80]:
# Los errores más frecuentes detrás de la fricción de la campaña
df.loc[df["status"] == "failed", "error_externo"].value_counts().head(10)

error_externo
(#132000) Number of parameters does not match the expected number of params         1374
131049: This message was not delivered to maintain healthy ecosystem engagement.       8
131026: Message undeliverable                                                          8
130472: User's number is part of an experiment                                         5
Template not found or invalid template name                                            1
Name: count, dtype: int64

In [81]:
# Qué plantillas concentran más fallas -> insumo directo para el diagnóstico de negocio
df.loc[df["status"] == "failed", "plantilla_nombre"].value_counts()

plantilla_nombre
lanzamiento_               1352
lanzamiento__inactivas       22
ampliaciones_zafiro          11
lanzamiento__activas_v2       8
saludo                        2
Name: count, dtype: int64

## 3. Clasificación básica (NLP ligero) de mensajes `incoming`

Clasificador por palabras clave sobre el texto que el cliente escribe. No es NLP con modelos (el caso pide
algo *ligero*, sin dependencias pesadas), pero sí resuelve la necesidad operativa: separar rápido "esto es
una queja" de "esto es un pedido".

**Diseño:**
- Diccionario de categoría → lista de palabras/frases gatillo, todo en minúsculas.
- Se evalúa en un orden de prioridad (`Queja` antes que `Saludo`, por ejemplo) para que un mensaje con
  varias señales caiga en la categoría más accionable primero.
- Mensajes sin texto (`NaN`) se marcan como `"Sin contenido"`, y lo que no matchea ninguna palabra clave
  queda como `"Otro"` — así el equipo puede revisar `"Otro"` periódicamente y engordar el diccionario.

**Limitación conocida:** al ser matching de substring, hay falsos positivos esperables (ej. "...estaremos
encantados de poder **ayuda**rte" cae en `Soporte` por la palabra "ayuda" dentro de "ayudarte"). Para
producción, el siguiente paso natural sería tokenizar por palabra completa en vez de substring, o pasar a
un clasificador entrenado — pero para el volumen y objetivo de este ejercicio, el approach por palabras
clave ya es accionable.

In [82]:
import pandas as pd
import unicodedata
import re


# ============================================================
# PALABRAS CLAVE
# ============================================================

PALABRAS_CLAVE = {

    "Queja": [
        "queja",
        "reclamo",
        "inconforme",
        "molest",
        "mal servicio",
        "no llego",
        "no llegó",
        "hizo falta",
        "faltó",
        "falto",
        "regad",
        "danad",
        "dañad",
        "mal estado",
        "roto",
        "equivocad",
        "cobra",
        "cobro",
        "estafa",
        "demora",
        "tarde",
        "no me gusta",
        "no me gustó",
        "no me gusto",

        # Problemas / acciones que no funcionan
        "no se deja",
        "no se deja entrar",
        "no se deja abrir",
        "no se deja descargar",
        "no se deja instalar",
        "no se deja actualizar",
        "no se deja iniciar",
        "no se deja ejecutar",
        "no se deja conectar",
        "no se deja sincronizar",
        "no se deja configurar",
        "no se deja personalizar",
        "no se deja modificar",
        "no se deja eliminar",
        "no se deja desinst"
    ],

    "Soporte": [
        "no puedo",
        "no pude",
        "error",
        "no funciona",
        "ayuda",
        "problema",
        "no entra",
        "no carga",
        "como hago",
        "cómo hago",
        "clave",
        "contraseña",
        "acceso",
        "pagina",
        "página",
        "app",
        "plataforma",

        # Solicitudes de ayuda
        "podrias",
        "podrías",
        "por favor",
        "puede",
        "puedes",
        "me puede",
        "me puedes",
        "hacer el favor",
        "favor",
        "mandar",
        "mandarme",
        "enviar",
        "enviarme",
        "revista",
        "informacion",
        "información",
        "pregunto",

        # Consultas generales
        "garantia",
        "garantía",
        "opciones"
    ],

    "Pedido": [
        "pedido",
        "pedir",
        "quiero comprar",
        "catalogo",
        "catálogo",
        "precio",
        "cuanto cuesta",
        "cuánto cuesta",
        "disponible",
        "comprar",
        "factura",
        "envio",
        "envío",
        "domicilio",
        "puedo comprar",
        "puedo pedir",
        "quiero pedir"
    ],

    "Saludo": [
        "hola",
        "buenas",
        "buenos dias",
        "buenos días",
        "buenas tardes",
        "buenas noches"
    ],

    "Despedida": [
        "adios",
        "adiós",
        "chau",
        "hasta luego",
        "nos vemos",
        "hasta pronto",
        "hasta mañana",
        "nos vemos luego",
        "gracias"
    ]
}


# ============================================================
# NORMALIZAR TEXTO
# ============================================================

def normalizar_texto(texto):
    """
    Normaliza el texto para facilitar la búsqueda:

    - Convierte a minúsculas
    - Elimina tildes
    - Elimina signos de puntuación
    - Elimina espacios repetidos
    """

    texto = str(texto).lower().strip()

    # Eliminar tildes
    texto = unicodedata.normalize("NFD", texto)

    texto = "".join(
        caracter
        for caracter in texto
        if unicodedata.category(caracter) != "Mn"
    )

    # Reemplazar signos de puntuación por espacios
    texto = re.sub(r"[^\w\s]", " ", texto)

    # Eliminar espacios repetidos
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto


# ============================================================
# CLASIFICAR MENSAJE
# ============================================================

def clasificar_mensaje(texto) -> str:

    # Si el mensaje está vacío
    if pd.isna(texto):
        return "Sin contenido"

    # Normalizar mensaje
    texto_normalizado = normalizar_texto(texto)

    # --------------------------------------------------------
    # PRIORIDAD DE CATEGORÍAS
    #
    # Primero buscamos:
    # 1. Queja
    # 2. Soporte
    # 3. Pedido
    # 4. Saludo
    # 5. Despedida
    #
    # Esto evita que "Buenos días" gane sobre una solicitud.
    # --------------------------------------------------------

    prioridades = [
        "Queja",
        "Soporte",
        "Pedido",
        "Saludo",
        "Despedida"
    ]

    for categoria in prioridades:

        for palabra in PALABRAS_CLAVE[categoria]:

            palabra_normalizada = normalizar_texto(palabra)

            # Detecta la palabra/frase en:
            # - inicio
            # - medio
            # - final
            if palabra_normalizada in texto_normalizado:
                return categoria

    # Si no encontró ninguna palabra clave
    return "Otro"


# ============================================================
# CREAR COLUMNA DE CATEGORÍA
# ============================================================

df["categoria_mensaje"] = pd.NA


# ============================================================
# FILTRAR SOLO MENSAJES ENTRANTES
# ============================================================

mask_incoming = df["message_type"] == "incoming"


# ============================================================
# CLASIFICAR MENSAJES
# ============================================================

df.loc[mask_incoming, "categoria_mensaje"] = (
    df.loc[mask_incoming, "content"]
    .apply(clasificar_mensaje)
)


# ============================================================
# MOSTRAR RESULTADOS
# ============================================================

resultado = (
    df.loc[
        mask_incoming,
        "categoria_mensaje"
    ]
    .value_counts()
)


print(resultado)

categoria_mensaje
Otro             186
Saludo            54
Soporte           52
Sin contenido     39
Pedido            38
Despedida         34
Queja             27
Name: count, dtype: int64


In [83]:
import datetime

# Convertir el resultado decimal a entero usando int()
semilla = int(datetime.datetime.now().second * 2.2)

# Aplicar el sample sin errores
df.loc[mask_incoming, ["content", "categoria_mensaje"]].sample(85, random_state=semilla)


,content,categoria_mensaje
2393,45957. 1,Otro
3650,Como estas,Otro
2714,Necesito hacer un acuerdo de pago,Otro
1467,Hola buenos días me puedes mandar los catálogos virtuales y las ofertas de e...,Soporte
1851,Hola buenas tardes,Queja
...,...,...
1301,Ok,Otro
1572,30569379,Otro
1154,Ok,Otro
2209,Buen día,Otro


In [84]:
df

,id,content,account_id,inbox_id,conversation_id,message_type,created_at,updated_at,private,status,...,sentiment,contact_id,sistema_lead_id,conversation_display_id,id_sistema,id_db_externa,id_leads_db_externa,error_externo,plantilla_nombre,categoria_mensaje
0,31888,La conversación fue marcada por Rojas bpo,37,78,2460,activity,2023-07-28 14:27:58,2023-07-28 14:27:58,False,sent,...,{},15090,16734,1486,31888,<NA>,<NA>,<NA>,<NA>,<NA>
1,30276,Gracias por comunicarse con nosotros. 💙¶¶Espero haber sido de ayuda.¶¶Le des...,37,78,5355,outgoing,2023-07-26 00:19:49,2023-07-26 00:19:51,False,delivered,...,{},14459,16103,4265,30276,<NA>,<NA>,<NA>,<NA>,<NA>
2,35098,"¡Hola, Johana Andrea Pascuas Perez! 💖¶🚀 ¡Llegó ! Un nuevo serie comienza con...",37,78,7226,outgoing,2023-07-29 13:43:11,2023-07-29 13:43:12,False,failed,...,{},20743,22387,6006,35098,<NA>,<NA>,(#132000) Number of parameters does not match the expected number of params,lanzamiento_,<NA>
3,37124,"¡Hola, Yeisy Yazmin Fino Galeano! 💖¶🚀 ¡Llegó ! Un nuevo serie comienza con g...",37,78,2324,outgoing,2023-07-29 13:56:47,2023-07-29 13:56:47,False,failed,...,{},17699,19343,1350,37124,<NA>,<NA>,(#132000) Number of parameters does not match the expected number of params,lanzamiento_,<NA>
4,41905,Asignado a SoporteTI por Default Policy,37,78,1168,activity,2023-07-29 19:19:05,2023-07-29 19:19:05,False,sent,...,{},14671,16315,194,41905,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3682,34011,"¡Hola, Lina Marcela Carvajal Garcia! 💖¶🚀 ¡Llegó ! Un nuevo serie comienza co...",37,78,5833,outgoing,2023-07-29 13:38:34,2023-07-29 13:38:35,False,failed,...,{},18960,20604,4694,34011,<NA>,<NA>,(#132000) Number of parameters does not match the expected number of params,lanzamiento_,<NA>
3683,32953,Me podrías hacer el favor de los catálogos en pdf de serie 12,37,78,5925,incoming,2023-07-28 21:09:34,2023-07-28 21:09:34,False,sent,...,{},15572,17216,4786,32953,<NA>,<NA>,<NA>,<NA>,Soporte
3684,35490,"¡Hola, Lucila Monroy Rodriguez! 💖¶🚀 ¡Llegó ! Un nuevo serie comienza con gra...",37,78,2707,outgoing,2023-07-29 13:45:17,2023-07-29 13:45:18,False,failed,...,{},18697,20341,1733,35490,<NA>,<NA>,(#132000) Number of parameters does not match the expected number of params,lanzamiento_,<NA>
3685,36540,"¡Hola, Luz Marina Mu?Oz! 💖¶🚀 ¡Llegó ! Un nuevo serie comienza con grandes op...",37,78,5612,outgoing,2023-07-29 13:52:12,2023-07-29 13:52:13,False,failed,...,{},16716,18360,4473,36540,<NA>,<NA>,(#132000) Number of parameters does not match the expected number of params,lanzamiento_,<NA>


## Exportar resultado

Guardamos el DataFrame limpio y enriquecido. Este `.csv` es el que se cargaría a PostgreSQL/MySQL
(vía `to_sql` o un `COPY`) para resolver el punto 2 del caso (las consultas SQL de funnel, SLA y
curva horaria).

In [85]:
df.to_csv("Data/Output/prueba_limpio.csv", index=False)
print("Archivo exportado: prueba_limpio.csv")
print(f"Shape final: {df.shape}")

Archivo exportado: prueba_limpio.csv
Shape final: (3687, 27)


## 4. Carga a MySQL — tabla para el punto 2

Este paso toma el DataFrame `df` ya limpio y enriquecido (el mismo que se exporta arriba a
`prueba_limpio.csv`) y lo sube directo a la base **pruebatecnica** en MySQL, así no dependes de
importar el CSV manualmente desde Workbench.

**Antes de correr esta celda:**

1. Crea un archivo `.env` en la misma carpeta que este notebook, con tus credenciales reales de Workbench:

```
DB_USER=root
DB_PASSWORD=tu_password_de_workbench
DB_HOST=127.0.0.1
DB_PORT=3306
DB_NAME=pruebatecnica
```

2. Instala el driver si aún no lo tienes: `pip install pymysql python-dotenv`.
3. Verifica que el servidor MySQL esté corriendo (Workbench conectado sin error).


In [86]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

DB_USER = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST", "127.0.0.1")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_NAME = os.getenv("DB_NAME", "pruebatecnica")

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Prueba rápida de conexión
with engine.connect() as conn:
    print("Conexión exitosa a la base:", DB_NAME)


Conexión exitosa a la base: pruebatecnica


In [87]:
NOMBRE_TABLA = "mensajes"

df.to_sql(NOMBRE_TABLA, engine, if_exists="replace", index=False, chunksize=1000)

print(f"Tabla '{NOMBRE_TABLA}' cargada en '{DB_NAME}' con {len(df)} filas.")


Tabla 'mensajes' cargada en 'pruebatecnica' con 3687 filas.


In [88]:
# Verificación: confirma en el notebook que los datos llegaron completos a MySQL
pd.read_sql(f"SELECT COUNT(*) AS total_filas FROM {NOMBRE_TABLA}", engine)


,total_filas
0,3687
